# KV Cache - Complete Implementation Guide

This notebook covers:
- **Basic Implementation**: Core concepts
- **Advanced Implementation**: Production patterns
- **Real-World Examples**: Industry implementations

Source: `llm/concepts/kv-cache.md`

## Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time

print("Libraries loaded")

## Basic Implementation

In [ ]:
# Basic KV Cache Implementation
import torch

class SimpleKVCache:
    """Store K,V from previous steps to avoid recomputation"""

    def __init__(self, max_seq_len=2048, hidden_dim=768, num_heads=12):
        self.cache = {
            'k': torch.zeros(1, max_seq_len, hidden_dim),
            'v': torch.zeros(1, max_seq_len, hidden_dim)
        }
        self.seq_pos = 0

    def update(self, k, v):
        """Add new K,V to cache"""
        seq_len = k.shape[1]
        self.cache['k'][:, self.seq_pos:self.seq_pos+seq_len] = k
        self.cache['v'][:, self.seq_pos:self.seq_pos+seq_len] = v
        self.seq_pos += seq_len

    def get_cached(self):
        """Get all cached K,V"""
        return self.cache['k'][:, :self.seq_pos], self.cache['v'][:, :self.seq_pos]

    def clear(self):
        """Clear cache for new sequence"""
        self.seq_pos = 0

# Usage
cache = SimpleKVCache()
for i in range(5):
    k = torch.randn(1, 1, 768)  # New token K
    v = torch.randn(1, 1, 768)  # New token V
    cache.update(k, v)

cached_k, cached_v = cache.get_cached()
print(f"Cached K shape: {cached_k.shape}")

## Advanced Implementation

In [ ]:
# Advanced KV Cache with Rotation and Quantization
import torch
import torch.nn.functional as F

class OptimizedKVCache:
    """Production KV cache with quantization and rope"""

    def __init__(self, max_seq_len=4096, hidden_dim=768, num_heads=12, dtype=torch.float16):
        self.max_seq_len = max_seq_len
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.dtype = dtype

        # Allocate cache as float16 for memory efficiency
        self.k_cache = torch.zeros(1, max_seq_len, hidden_dim, dtype=dtype)
        self.v_cache = torch.zeros(1, max_seq_len, hidden_dim, dtype=dtype)
        self.seq_pos = 0

    def update(self, k, v, start_pos):
        """Update cache at position"""
        seq_len = k.shape[1]

        # Store in cache
        self.k_cache[:, start_pos:start_pos+seq_len] = k.to(self.dtype)
        self.v_cache[:, start_pos:start_pos+seq_len] = v.to(self.dtype)

        return self.k_cache[:, :start_pos+seq_len], self.v_cache[:, :start_pos+seq_len]

    def get_size_mb(self):
        """Get cache memory usage"""
        dtype_size = 2 if self.dtype == torch.float16 else 4  # bytes
        size_bytes = self.k_cache.numel() * dtype_size * 2  # k + v
        return size_bytes / (1024 * 1024)

# Usage
cache = OptimizedKVCache(max_seq_len=4096, dtype=torch.float16)
print(f"Cache size: {cache.get_size_mb():.1f} MB")

# Simulate generation
for pos in range(100):
    k = torch.randn(1, 1, 768)
    v = torch.randn(1, 1, 768)
    cached_k, cached_v = cache.update(k, v, pos)

## Real-World: Flash

In [ ]:
# Real-World: Flash Attention with KV Cache
class FlashAttentionKVCache:
    """Flash Attention optimized with KV cache"""

    def __init__(self, block_size=128):
        self.block_size = block_size
        self.cache = {'k': [], 'v': []}

    def flash_attention(self, q, k_cached, v_cached):
        """Flash attention: process in blocks"""
        # Block-wise computation for memory efficiency
        outputs = []

        block_size = self.block_size
        for i in range(0, k_cached.shape[1], block_size):
            k_block = k_cached[:, i:i+block_size]
            v_block = v_cached[:, i:i+block_size]

            # Compute attention for block
            scores = torch.matmul(q, k_block.transpose(-2, -1))
            scores = torch.softmax(scores / (q.shape[-1] ** 0.5), dim=-1)
            block_out = torch.matmul(scores, v_block)
            outputs.append(block_out)

        return torch.cat(outputs, dim=1).sum(dim=1, keepdim=True)

    def add_to_cache(self, k, v):
        """Add new K,V to cache"""
        self.cache['k'].append(k)
        self.cache['v'].append(v)

    def get_cache(self):
        """Get full cache"""
        k_full = torch.cat(self.cache['k'], dim=1) if self.cache['k'] else None
        v_full = torch.cat(self.cache['v'], dim=1) if self.cache['v'] else None
        return k_full, v_full

# Usage
cache = FlashAttentionKVCache()
q = torch.randn(2, 1, 768)

for i in range(100):
    k = torch.randn(2, 1, 768)
    v = torch.randn(2, 1, 768)
    cache.add_to_cache(k, v)

    k_full, v_full = cache.get_cache()
    if k_full is not None:
        output = cache.flash_attention(q, k_full, v_full)

## Real-World: Llama

In [ ]:
# Real-World: LLAMA KV Cache Strategy
class LLAMAKVCache:
    """LLAMA's rotary positional embedding with KV cache"""

    def __init__(self, max_batch_size=32, max_seq_len=4096, hidden_dim=4096, num_heads=32):
        self.max_batch_size = max_batch_size
        self.max_seq_len = max_seq_len
        self.head_dim = hidden_dim // num_heads

        # Allocate cache for batch
        self.k_cache = torch.zeros(max_batch_size, max_seq_len, hidden_dim)
        self.v_cache = torch.zeros(max_batch_size, max_seq_len, hidden_dim)

    def attention_with_cache(self, q, k, v, start_pos, seq_len):
        """Attention with cached K,V"""
        batch_size = q.shape[0]

        # Update cache with new K,V
        self.k_cache[:batch_size, start_pos:start_pos+seq_len] = k
        self.v_cache[:batch_size, start_pos:start_pos+seq_len] = v

        # Use cached K,V for attention
        cached_k = self.k_cache[:batch_size, :start_pos+seq_len]
        cached_v = self.v_cache[:batch_size, :start_pos+seq_len]

        # Compute attention
        scores = torch.matmul(q, cached_k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn, cached_v)

        return output

# Usage simulating LLAMA generation
cache = LLAMAKVCache()
for step in range(10):
    q = torch.randn(1, 1, 4096)  # Query for current token
    k = torch.randn(1, 1, 4096)  # Key for current token
    v = torch.randn(1, 1, 4096)  # Value for current token

    output = cache.attention_with_cache(q, k, v, start_pos=step, seq_len=1)
    print(f"Step {step}: output shape {output.shape}")

## Resources

- **Markdown**: `llm/concepts/kv-cache.md`
- **Interview Q&A**: See markdown file
- **Real-world**: Review code above
- **Next Steps**: Try modifying the code